# DeBERTa Score-Only Abstract Evaluator (Modular)

This notebook mirrors the qwen experiment structure, but fine-tunes DeBERTa for score prediction only (no rationale generation).

It supports:
- train/val/test load + clean/validate
- score-only text construction
- HF datasets + tokenization
- LoRA (or frozen-base) fine-tuning
- validation/test prediction + metrics export


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "experiments").exists() and (p / "data").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
import sys
import importlib

print("Python executable:", sys.executable)

required_base = {
    "torch": "torch",
    "transformers": "transformers",
    "peft": "peft",
    "sentencepiece": "sentencepiece",
}

optional_any_of = {
    "protobuf": "google.protobuf",
    "tiktoken": "tiktoken",
}

missing = []
versions = {}

for pip_name, module_name in required_base.items():
    try:
        mod = importlib.import_module(module_name)
        versions[pip_name] = getattr(mod, "__version__", "unknown")
    except Exception:
        missing.append(pip_name)

found_optional = []
for pip_name, module_name in optional_any_of.items():
    try:
        mod = importlib.import_module(module_name)
        versions[pip_name] = getattr(mod, "__version__", "unknown")
        found_optional.append(pip_name)
    except Exception:
        pass

print("Package versions:", versions)
if missing:
    raise RuntimeError(
        "Missing packages: " + ", ".join(missing) + "\n"
        "Install with: pip install " + " ".join(missing)
    )

if not found_optional:
    raise RuntimeError(
        "Need at least one tokenizer backend helper: protobuf or tiktoken\n"
        "Install with: pip install protobuf\n"
        "(or: pip install tiktoken)"
    )


In [ ]:
import json

from experiments.deberta.utils.configs import build_data_paths, build_deberta_default_config
from experiments.deberta.utils.data import add_score_only_text, to_hf_dataset_dict
from experiments.deberta.utils.modeling import (
    build_data_collator,
    load_deberta_score_model,
    load_tokenizer,
)
from experiments.deberta.utils.pipeline import (
    predict_split,
    save_predictions_and_metrics,
    tokenize_dataset_dict,
)
from experiments.deberta.utils.training import train_deberta

from experiments.utils.data import (
    clean_train_val_test,
    load_train_val_test_dfs,
    score_distribution,
)
from experiments.utils.logging_utils import setup_logger
from experiments.utils.runtime import configure_wandb_dir, set_global_seed


In [ ]:
cfg = build_deberta_default_config(PROJECT_ROOT)

# --- Core run identity ---
cfg.model_name = "microsoft/deberta-v3-base"
cfg.run_name = "deberta_v3_base_abstract_evaluator_lora_score_only_3e-5"

# --- Data paths (edit if you want different splits) ---
cfg.data_paths = build_data_paths(
    train_path=PROJECT_ROOT / "data/data/train/all.jsonl",
    val_path=PROJECT_ROOT / "data/data/val/all.jsonl",
    test_path=PROJECT_ROOT / "data/data/test/all.jsonl",
)

# --- Model/trainer config ---
cfg.max_length = 512
cfg.num_labels = 5
cfg.train.learning_rate = 3e-5
cfg.train.num_train_epochs = 6
cfg.train.per_device_train_batch_size = 8
cfg.train.per_device_eval_batch_size = 16
cfg.train.gradient_accumulation_steps = 1
cfg.train.eval_strategy = "epoch"
cfg.train.save_strategy = "epoch"
cfg.train.metric_for_best_model = "eval_mae"
cfg.train.greater_is_better = False

# LoRA on/off
cfg.lora.enabled = True
cfg.lora.r = 16
cfg.lora.lora_alpha = 32
cfg.lora.lora_dropout = 0.1

# W&B logging on/off
cfg.wandb_enabled = False

cfg


In [ ]:
set_global_seed(cfg.seed)

if cfg.wandb_dir is not None:
    configure_wandb_dir(str(cfg.wandb_dir))

log_dir = cfg.output_root / "logs"
logger = setup_logger(
    name=f"{cfg.run_name}_pipeline",
    log_dir=log_dir,
    log_file=f"{cfg.run_name}.log",
)
logger.info("Initialized run config: %s", json.dumps(cfg.as_dict(), ensure_ascii=False))
log_dir


In [ ]:
train_df, val_df, test_df = load_train_val_test_dfs(
    train_path=cfg.data_paths.train_path,
    val_path=cfg.data_paths.val_path,
    test_path=cfg.data_paths.test_path,
)

train_df, val_df, test_df = clean_train_val_test(train_df, val_df, test_df)

print("Shapes:", train_df.shape, val_df.shape, test_df.shape)
print("Train score dist:", score_distribution(train_df))
print("Val score dist:", score_distribution(val_df))
print("Test score dist:", score_distribution(test_df))


In [ ]:
train_df = add_score_only_text(train_df)
val_df = add_score_only_text(val_df)
test_df = add_score_only_text(test_df)

print(train_df[["text", "score"]].head(1).to_dict("records")[0]["text"][:1500])

ds = to_hf_dataset_dict(train_df, val_df, test_df)

tokenizer = load_tokenizer(cfg.model_name)
ds_tok = tokenize_dataset_dict(ds, tokenizer=tokenizer, max_length=cfg.max_length)
collator = build_data_collator(tokenizer)

print(ds_tok)


In [ ]:
model = load_deberta_score_model(
    model_name=cfg.model_name,
    num_labels=cfg.num_labels,
    use_lora=cfg.lora.enabled,
    lora_r=cfg.lora.r,
    lora_alpha=cfg.lora.lora_alpha,
    lora_dropout=cfg.lora.lora_dropout,
    lora_target_modules=cfg.lora.target_modules,
    lora_bias=cfg.lora.bias,
    freeze_lower_n_when_no_lora=cfg.lora.freeze_lower_n_when_no_lora,
)

model


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    trainer, run_info = train_deberta(
        cfg=cfg,
        ds=ds_tok,
        model=model,
        tokenizer=tokenizer,
    )
    print(run_info)
else:
    print("RUN_TRAINING=False -> training skipped.")


In [ ]:
RUN_EVAL = False

if RUN_EVAL:
    from pathlib import Path
    from transformers import AutoModelForSequenceClassification, Trainer

    if "run_info" not in locals():
        run_info_path = cfg.output_root / "models" / cfg.run_name / "run_info.json"
        if not run_info_path.exists():
            raise FileNotFoundError(f"run_info.json not found: {run_info_path}")
        run_info = json.loads(run_info_path.read_text(encoding="utf-8"))

    if "trainer" not in locals():
        best_model_dir = Path(run_info["best_model_dir"])

        if cfg.lora.enabled:
            from peft import PeftModel

            base_model = AutoModelForSequenceClassification.from_pretrained(
                cfg.model_name,
                num_labels=cfg.num_labels,
                id2label={i: str(i) for i in range(cfg.num_labels)},
                label2id={str(i): i for i in range(cfg.num_labels)},
                problem_type="single_label_classification",
            )
            loaded_model = PeftModel.from_pretrained(base_model, str(best_model_dir))
        else:
            loaded_model = AutoModelForSequenceClassification.from_pretrained(str(best_model_dir))

        trainer = Trainer(model=loaded_model, tokenizer=tokenizer, data_collator=collator)

    val_pred_df, val_metrics = predict_split(trainer, ds_tok["validation"], val_df)
    test_pred_df, test_metrics = predict_split(trainer, ds_tok["test"], test_df)

    val_saved = save_predictions_and_metrics(cfg.output_root, cfg.run_name, "validation", val_pred_df, val_metrics)
    test_saved = save_predictions_and_metrics(cfg.output_root, cfg.run_name, "test", test_pred_df, test_metrics)

    print(json.dumps({
        "validation_metrics": val_metrics,
        "test_metrics": test_metrics,
        "validation_files": val_saved,
        "test_files": test_saved,
    }, indent=2))
else:
    print("RUN_EVAL=False -> evaluation/prediction skipped.")
